# audio

> Audio file serving route handlers for alignment playback

In [ ]:
#| default_exp routes.core.audio

In [ ]:
#| export
from pathlib import Path
from typing import Tuple, Dict, Callable

from fasthtml.common import APIRouter
from starlette.responses import FileResponse, Response

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

DEBUG_AUDIO = False

## Handlers

In [ ]:
#| export
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
):  # Audio file response or 404
    """Serve the audio file for the current alignment session."""
    session_id = get_session_id(sess)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Serving audio for session: {session_id}")

    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    align_state = state.get("step_states", {}).get("alignment", {})
    media_path = align_state.get("media_path")

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] media_path from state: {media_path}")
        if media_path:
            print(f"[AUDIO_SRC] file exists: {Path(media_path).is_file()}")

    if not media_path or not Path(media_path).is_file():
        if DEBUG_AUDIO:
            print(f"[AUDIO_SRC] Returning 404 - media_path missing or file not found")
        return Response(status_code=404)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Returning FileResponse for: {media_path}")

    return FileResponse(str(media_path), media_type="audio/mpeg")

## Router

In [ ]:
#| export
def init_audio_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/audio")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize audio serving routes."""
    router = APIRouter(prefix=prefix)

    @router
    def audio_src(request, sess):
        """Serve the audio file for alignment playback."""
        return _handle_audio_src(workflow, sess)

    routes = {
        "audio_src": audio_src,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()